<a href="https://colab.research.google.com/github/SieCiRou/photo_take/blob/main/photo_take.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/Camera_Continuous', exist_ok=True)

In [ ]:
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import io
import time
import os
from datetime import datetime
from google.colab import drive
from google.colab import output

# ====================== 設定區 ======================
SAVE_FOLDER = '/content/drive/MyDrive/Camera_Continuous/'  # 儲存路徑
DURATION = 23          # 總拍攝秒數 (可自行修改)
INTERVAL = 2.0         # 每幾秒拍一張 (0.5 = 每0.5秒一張)
QUALITY = 0.99        # 畫質 0.0~1.0
CAMERA_INDEX = 1
# ===================================================

# 先掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 建立資料夾
os.makedirs(SAVE_FOLDER, exist_ok=True)

def continuous_capture():
  js = Javascript('''
  let stream = null;
  let timer = null;
  let count = 0;

  async function startContinuous(duration, interval) {
    const div = document.createElement('div');
    div.style.position = 'fixed';
    div.style.top = '10px';
    div.style.left = '10px';
    div.style.background = 'white';
    div.style.padding = '10px';
    div.style.border = '2px solid black';
    div.style.zIndex = 9999;

    const video = document.createElement('video');
    video.style.width = '520px';
    div.appendChild(video);

    const status = document.createElement('div');
    status.style.marginTop = '8px';
    div.appendChild(status);

    const stopBtn = document.createElement('button');
    stopBtn.textContent = '🛑 停止拍攝';
    stopBtn.style.marginLeft = '10px';
    div.appendChild(stopBtn);

    document.body.appendChild(div);

    stream = await navigator.mediaDevices.getUserMedia({video: {
    width: { ideal: 1920, min: 1280 },   // 優先要 1080p 或 720p
    height: { ideal: 1080, min: 720 },
    frameRate: { ideal: 30, max: 30 },   // 固定 30fps，避免有些相機會自動降解析度
    facingMode: "user"                    // 前置鏡頭（筆電通常是這個）
  }});
    video.srcObject = stream;
    await video.play();

    // 等 video 真正拿到解析度
    await new Promise(resolve => {
      if (video.videoWidth > 0) resolve();
      else video.onloadedmetadata = () => resolve();
    });
    const canvas = document.createElement('canvas');
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;

    async function captureFrame() {
      count++;
      const now = new Date().toISOString().slice(11,19).replace(/:/g,'');
      const filename = `snapshot_${now}_${count}.jpg`;

      canvas.getContext('2d').drawImage(video, 0, 0);
      const dataUrl = canvas.toDataURL('image/jpeg', 0.98);

      // 傳回 Python
      google.colab.kernel.invokeFunction('notebook.capture', [dataUrl, filename], {});

      status.textContent = `已拍第 ${count} 張 | 剩餘 ${Math.max(0, Math.ceil((duration*1000 - (Date.now()-startTime))/1000))} 秒`;
    }

    const startTime = Date.now();
    timer = setInterval(captureFrame, interval * 1000);

    // 總時間結束自動停止
    setTimeout(() => {
        stopCapture();
    }, duration * 1000);

    // 手動停止
    stopBtn.onclick = stopCapture;

    function stopCapture() {
        clearInterval(timer);
        if (stream) stream.getTracks().forEach(track => track.stop());
        div.remove();
        google.colab.kernel.invokeFunction('notebook.stop', [], {});
    }
  }
  ''')

  display(js)
  eval_js(f'startContinuous({DURATION}, {INTERVAL})')

# ==================== Python 接收端 ====================
from google.colab import output

captured_count = 0

def save_image(data_url, filename):
    global captured_count
    captured_count += 1
    binary = b64decode(data_url.split(',')[1])
    img = Image.open(io.BytesIO(binary))

    save_path = os.path.join(SAVE_FOLDER, filename)
    img.save(save_path)

    print(f'✅ 已儲存: {filename}  (第 {captured_count} 張)', end='\r')

def stop_capture():
    print(f'\n 拍攝結束！共儲存 {captured_count} 張照片')
    print(f' 儲存位置：{SAVE_FOLDER}')

# 註冊 Python 函數給 JavaScript 呼叫
output.register_callback('notebook.capture', save_image)
output.register_callback('notebook.stop', stop_capture)

# ====================== 開始拍攝 ======================
print("即將開始連續拍攝...")
print(f" ====== 總時長：{DURATION} 秒 | 間隔：{INTERVAL} 秒/張 ======= ")
continuous_capture()

Mounted at /content/drive
即將開始連續拍攝...
 ====== 總時長：23 秒 | 間隔：2.0 秒/張 ======= 


<IPython.core.display.Javascript object>


 拍攝結束！共儲存 11 張照片
 儲存位置：/content/drive/MyDrive/Camera_Continuous/


In [ ]:
# 單張拍照
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import io
import time
import os
from datetime import datetime

# ====================== 設定區 ======================
SAVE_FOLDER = '/content/drive/MyDrive/Camera_Continuous/'  # 儲存路徑
DURATION = 3          # 總拍攝秒數 (可自行修改)
INTERVAL = 2.0         # 每幾秒拍一張 (0.5 = 每0.5秒一張)
QUALITY = 0.99        # 畫質 0.0~1.0
CAMERA_INDEX = 1
MAX_IMAGES = 10
# ===================================================

# 先掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 建立資料夾
os.makedirs(SAVE_FOLDER, exist_ok=True)

def continuous_capture():
  js = Javascript('''
  let stream = null;
  let timer = null;
  let count = 0;

  async function startContinuous(maxImages, interval) {
    const div = document.createElement('div');
    div.style.position = 'fixed';
    div.style.top = '10px';
    div.style.left = '10px';
    div.style.background = 'white';
    div.style.padding = '10px';
    div.style.border = '2px solid black';
    div.style.zIndex = 9999;

    const video = document.createElement('video');
    video.style.width = '520px';
    div.appendChild(video);

    const status = document.createElement('div');
    status.style.marginTop = '8px';
    div.appendChild(status);

    const stopBtn = document.createElement('button');
    stopBtn.textContent = '🛑 停止拍攝';
    stopBtn.style.marginLeft = '10px';
    div.appendChild(stopBtn);

    document.body.appendChild(div);

    stream = await navigator.mediaDevices.getUserMedia({video: {
    width: { ideal: 1920, min: 1280 },   // 優先要 1080p 或 720p
    height: { ideal: 1080, min: 720 },
    frameRate: { ideal: 30, max: 30 },   // 固定 30fps，避免有些相機會自動降解析度
    facingMode: "user"                    // 前置鏡頭（筆電通常是這個）
  }});
    video.srcObject = stream;
    await video.play();

    // 等 video 真正拿到解析度
    await new Promise(resolve => {
      if (video.videoWidth > 0) resolve();
      else video.onloadedmetadata = () => resolve();
    });
    const canvas = document.createElement('canvas');
    canvas.width = video.videoWidth;
    canvas.height = video.videoHeight;

    async function captureFrame() {
      if (count>=maxImages){
        stopCapture();
        return;
      }
      count++;
      const now = new Date().toISOString().slice(11,19).replace(/:/g,'');
      const filename = `snapshot_${now}_${count}.jpg`;

      canvas.getContext('2d').drawImage(video, 0, 0);
      const dataUrl = canvas.toDataURL('image/jpeg', 0.98);

      // 傳回 Python
      google.colab.kernel.invokeFunction('notebook.capture', [dataUrl, filename], {});

      status.textContent = `已拍第 ${count} 張 / 共 ${maxImages} 張 | 間隔: ${interval}秒`;
    }

    const startTime = Date.now();
    timer = setInterval(captureFrame, interval * 1000);

    // 手動停止
    stopBtn.onclick = stopCapture;

    function stopCapture() {
        if (timer) clearInterval(timer);
        if (stream) stream.getTracks().forEach(track => track.stop());
        div.remove();
        google.colab.kernel.invokeFunction('notebook.stop', [], {});
    }
  }
  ''')

  display(js)
  eval_js(f'startContinuous({DURATION}, {INTERVAL})')

# ==================== Python 接收端 ====================
from google.colab import output

captured_count = 0

def save_image(data_url, filename):
    global captured_count
    captured_count += 1
    binary = b64decode(data_url.split(',')[1])
    img = Image.open(io.BytesIO(binary))

    save_path = os.path.join(SAVE_FOLDER, filename)
    img.save(save_path)

    print(f'✅ 已儲存: {filename}  (第 {captured_count} 張)', end='\r')

def stop_capture():
    print(f'\n 拍攝結束！共儲存 {captured_count} 張照片')
    print(f' 儲存位置：{SAVE_FOLDER}')

# 註冊 Python 函數給 JavaScript 呼叫
output.register_callback('notebook.capture', save_image)
output.register_callback('notebook.stop', stop_capture)

# ====================== 開始拍攝 ======================
print("即將開始連續拍攝...")
print(f" ====== 總時長：{DURATION} 秒 | 間隔：{INTERVAL} 秒/張 ======= ")
continuous_capture()

In [ ]:
# 第一此需要同意並勾選記住相機，後面無須重複同意確認
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import io
import os
from google.colab import drive
from google.colab import output

# ====================== 設定區 ======================
AUTO_SELECT = 1
TARGET_CAMERA_NAME = "iVCam" # 建議縮短關鍵字，增加匹配成功率
MAX_IMAGES = 5
INTERVAL = 2.0
SAVE_FOLDER = '/content/drive/MyDrive/Camera_Continuous/'
# ===================================================

drive.mount('/content/drive', force_remount=True)
os.makedirs(SAVE_FOLDER, exist_ok=True)

def continuous_capture():
    js = Javascript('''
    async function startContinuous(maxImages, interval, autoSelect, targetName) {
        let stream = null;
        let deviceId = null;

        // 1. 喚醒權限：先隨便請求一個鏡頭，讓瀏覽器解鎖設備名稱
        try {
            const tempStream = await navigator.mediaDevices.getUserMedia({ video: true });
            tempStream.getTracks().forEach(track => track.stop()); // 隨即關閉，僅為了拿權限
        } catch (e) {
            alert("請先允許攝影機存取權限，否則無法選取特定鏡頭。");
            return;
        }

        // 2. 獲取設備列表 (此時 label 應該不再是空白了)
        if (autoSelect === 1) {
            const devices = await navigator.mediaDevices.enumerateDevices();
            const videoDevices = devices.filter(device => device.kind === 'videoinput');

            // 偵錯用：在瀏覽器 Console 印出所有找到的鏡頭名稱
            console.log("發現鏡頭列表:", videoDevices.map(d => d.label));

            const target = videoDevices.find(d =>
                d.label.toLowerCase().includes(targetName.toLowerCase())
            );

            if (target) {
                deviceId = target.deviceId;
                console.log("成功鎖定目標鏡頭: " + target.label);
            } else {
                console.warn("找不到匹配名稱的鏡頭，將由系統預設。");
            }
        }

        // 3. 建立 UI
        const div = document.createElement('div');
        div.style.cssText = 'position:fixed;top:10px;left:10px;background:white;padding:10px;border:2px solid black;z-index:9999;';
        const video = document.createElement('video');
        video.style.width = '520px';
        const status = document.createElement('div');
        status.style.marginTop = '8px';
        const stopBtn = document.createElement('button');
        stopBtn.textContent = '停止拍攝';
        div.appendChild(video);
        div.appendChild(status);
        div.appendChild(stopBtn);
        document.body.appendChild(div);

        // 4. 正式啟動目標鏡頭
        const constraints = {
            video: deviceId ? { deviceId: { exact: deviceId }, width: {ideal: 1920}, height: {ideal: 1080} }
                            : { width: {ideal: 1920}, height: {ideal: 1080} }
        };

        stream = await navigator.mediaDevices.getUserMedia(constraints);
        video.srcObject = stream;
        await video.play();

        const canvas = document.createElement('canvas');
        await new Promise(resolve => {
            if (video.videoWidth > 0) resolve();
            else video.onloadedmetadata = () => resolve();
        });
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;

        let count = 0;
        let timer = setInterval(async () => {
            if (count >= maxImages) {
                stopCapture();
                return;
            }
            count++;
            const now = new Date().toISOString().slice(11, 19).replace(/:/g, '');
            canvas.getContext('2d').drawImage(video, 0, 0);
            const dataUrl = canvas.toDataURL('image/jpeg', 0.98);
            google.colab.kernel.invokeFunction('notebook.capture', [dataUrl, `snap_${now}_${count}.jpg`], {});
            status.textContent = "已拍攝: " + count + " / " + maxImages;
        }, interval * 1000);

        function stopCapture() {
            clearInterval(timer);
            if (stream) stream.getTracks().forEach(track => track.stop());
            div.remove();
            google.colab.kernel.invokeFunction('notebook.stop', [], {});
        }
        stopBtn.onclick = stopCapture;
    }
    ''')

    display(js)
    eval_js(f'startContinuous({MAX_IMAGES}, {INTERVAL}, {AUTO_SELECT}, "{TARGET_CAMERA_NAME}")')

# --- Python 接收端 ---
captured_count = 0
def save_image(data_url, filename):
    global captured_count
    captured_count += 1
    binary = b64decode(data_url.split(',')[1])
    img = Image.open(io.BytesIO(binary))
    img.save(os.path.join(SAVE_FOLDER, filename))
    print(f'已儲存: {filename} (第 {captured_count} 張)', end='\r')

def stop_capture():
    print(f'\n拍攝結束，共儲存 {captured_count} 張照片至 {SAVE_FOLDER}')

output.register_callback('notebook.capture', save_image)
output.register_callback('notebook.stop', stop_capture)

captured_count = 0
continuous_capture()

Mounted at /content/drive


<IPython.core.display.Javascript object>


拍攝結束，共儲存 5 張照片至 /content/drive/MyDrive/Camera_Continuous/


In [ ]:
# 指定你要存的路徑（可自行修改）
photo_path = '/content/drive/MyDrive/Camera_Snapshots/snapshot_001.jpg'

# 執行拍照
take_photo(filename=photo_path, quality=0.95)

NameError: name 'take_photo' is not defined